In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
DATA_PATH = "/content/drive/MyDrive/hotel_bookings.csv"

assert os.path.exists(DATA_PATH), f"Dataset not found: {DATA_PATH}"

df_raw = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Raw dataset shape:", df_raw.shape)

display(df_raw.head())

Dataset loaded successfully.
Raw dataset shape: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [ ]:
data_dictionary = pd.DataFrame([
    ["hotel", "Type of hotel", "Categorical", "None", "City Hotel / Resort Hotel"],
    ["is_canceled", "Booking cancellation indicator", "Binary", "None", "0 or 1"],
    ["lead_time", "Days between booking and arrival", "Integer", "days", ">= 0"],
    ["arrival_date_year", "Arrival year", "Integer", "year", "2015-2017"],
    ["arrival_date_month", "Arrival month", "Categorical", "month", "January-December"],
    ["arrival_date_week_number", "Arrival week number", "Integer", "week", "1-53"],
    ["arrival_date_day_of_month", "Arrival day of month", "Integer", "day", "1-31"],
    ["stays_in_weekend_nights", "Weekend nights stayed", "Integer", "nights", ">= 0"],
    ["stays_in_week_nights", "Week nights stayed", "Integer", "nights", ">= 0"],
    ["adults", "Number of adults", "Integer", "guests", ">= 0"],
    ["children", "Number of children", "Numeric", "guests", ">= 0"],
    ["babies", "Number of babies", "Integer", "guests", ">= 0"],
    ["meal", "Meal type booked", "Categorical", "None", "Dataset categories"],
    ["country", "Guest country", "Categorical", "Country code", "Valid country codes"],
    ["market_segment", "Market segment", "Categorical", "None", "Dataset categories"],
    ["distribution_channel", "Booking distribution channel", "Categorical", "None", "Dataset categories"],
    ["is_repeated_guest", "Repeated guest indicator", "Binary", "None", "0 or 1"],
    ["previous_cancellations", "Previous cancelled bookings", "Integer", "bookings", ">= 0"],
    ["previous_bookings_not_canceled", "Previous non-cancelled bookings", "Integer", "bookings", ">= 0"],
    ["reserved_room_type", "Room type originally reserved", "Categorical", "None", "Dataset categories"],
    ["assigned_room_type", "Room type finally assigned", "Categorical", "None", "Dataset categories"],
    ["booking_changes", "Number of booking changes", "Integer", "changes", ">= 0"],
    ["deposit_type", "Deposit type", "Categorical", "None", "Dataset categories"],
    ["agent", "Travel agent identifier", "Categorical identifier", "ID", "Non-negative identifier"],
    ["company", "Company identifier", "Categorical identifier", "ID", "Non-negative identifier"],
    ["days_in_waiting_list", "Days on waiting list", "Integer", "days", ">= 0"],
    ["customer_type", "Customer type", "Categorical", "None", "Dataset categories"],
    ["adr", "Average daily rate", "Numeric", "currency/day", ">= 0"],
    ["required_car_parking_spaces", "Required parking spaces", "Integer", "spaces", ">= 0"],
    ["total_of_special_requests", "Number of special requests", "Integer", "requests", ">= 0"],
    ["reservation_status", "Final reservation status", "Categorical", "None", "Dataset categories"],
    ["reservation_status_date", "Date of final reservation status", "Date", "date", "Valid date"]
], columns=[
    "column",
    "meaning",
    "type",
    "unit",
    "expected_range"
])

print("Number of columns documented:", len(data_dictionary))

display(data_dictionary)

Number of columns documented: 32


,column,meaning,type,unit,expected_range
0,hotel,Type of hotel,Categorical,None,City Hotel / Resort Hotel
1,is_canceled,Booking cancellation indicator,Binary,None,0 or 1
2,lead_time,Days between booking and arrival,Integer,days,>= 0
3,arrival_date_year,Arrival year,Integer,year,2015-2017
4,arrival_date_month,Arrival month,Categorical,month,January-December
5,arrival_date_week_number,Arrival week number,Integer,week,1-53
6,arrival_date_day_of_month,Arrival day of month,Integer,day,1-31
7,stays_in_weekend_nights,Weekend nights stayed,Integer,nights,>= 0
8,stays_in_week_nights,Week nights stayed,Integer,nights,>= 0
9,adults,Number of adults,Integer,guests,>= 0


In [ ]:
raw_missing = df_raw.isna().sum()

raw_missing_count = int(raw_missing.sum())

raw_duplicates = int(
    df_raw.duplicated().sum()
)

print("Raw dataset shape:", df_raw.shape)
print("Total missing values:", raw_missing_count)
print("Duplicate rows:", raw_duplicates)

missing_report = pd.DataFrame({
    "column": df_raw.columns,
    "missing_count": raw_missing.values,
    "missing_percentage": (
        raw_missing.values / len(df_raw) * 100
    ).round(2)
})

display(
    missing_report[
        missing_report["missing_count"] > 0
    ].sort_values(
        "missing_count",
        ascending=False
    )
)

Raw dataset shape: (119390, 32)
Total missing values: 129425
Duplicate rows: 31994


,column,missing_count,missing_percentage
24,company,112593,94.31
23,agent,16340,13.69
13,country,488,0.41
10,children,4,0.00


In [ ]:
quality_checks = []

quality_checks.append([
    "negative_lead_time",
    int((df_raw["lead_time"] < 0).sum())
])

quality_checks.append([
    "invalid_arrival_week",
    int(
        (~df_raw["arrival_date_week_number"].between(1, 53)).sum()
    )
])

quality_checks.append([
    "invalid_arrival_day",
    int(
        (~df_raw["arrival_date_day_of_month"].between(1, 31)).sum()
    )
])

quality_checks.append([
    "negative_weekend_nights",
    int(
        (df_raw["stays_in_weekend_nights"] < 0).sum()
    )
])

quality_checks.append([
    "negative_week_nights",
    int(
        (df_raw["stays_in_week_nights"] < 0).sum()
    )
])

quality_checks.append([
    "negative_adults",
    int(
        (df_raw["adults"] < 0).sum()
    )
])

quality_checks.append([
    "negative_children",
    int(
        (df_raw["children"].dropna() < 0).sum()
    )
])

quality_checks.append([
    "negative_babies",
    int(
        (df_raw["babies"] < 0).sum()
    )
])

quality_checks.append([
    "negative_adr",
    int(
        (df_raw["adr"] < 0).sum()
    )
])

quality_checks.append([
    "invalid_is_canceled",
    int(
        (~df_raw["is_canceled"].isin([0, 1])).sum()
    )
])

quality_checks.append([
    "invalid_is_repeated_guest",
    int(
        (~df_raw["is_repeated_guest"].isin([0, 1])).sum()
    )
])

quality_checks.append([
    "zero_total_guests",
    int(
        (
            (df_raw["adults"].fillna(0) == 0) &
            (df_raw["children"].fillna(0) == 0) &
            (df_raw["babies"].fillna(0) == 0)
        ).sum()
    )
])

quality_report = pd.DataFrame(
    quality_checks,
    columns=["problem", "count"]
)

display(quality_report)

,problem,count
0,negative_lead_time,0
1,invalid_arrival_week,0
2,invalid_arrival_day,0
3,negative_weekend_nights,0
4,negative_week_nights,0
5,negative_adults,0
6,negative_children,0
7,negative_babies,0
8,negative_adr,1
9,invalid_is_canceled,0


In [ ]:
category_columns = [
    "hotel",
    "meal",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "assigned_room_type",
    "deposit_type",
    "customer_type",
    "reservation_status"
]

category_consistency = []

for column in category_columns:

    values = df_raw[column].dropna().astype(str)

    stripped = values.str.strip()

    category_consistency.append([
        column,
        int((values != stripped).sum()),
        int(
            values.str.lower().nunique() != values.nunique()
        )
    ])

category_consistency = pd.DataFrame(
    category_consistency,
    columns=[
        "column",
        "leading_or_trailing_spaces",
        "case_variation_indicator"
    ]
)

display(category_consistency)

,column,leading_or_trailing_spaces,case_variation_indicator
0,hotel,0,0
1,meal,0,0
2,market_segment,0,0
3,distribution_channel,0,0
4,reserved_room_type,0,0
5,assigned_room_type,0,0
6,deposit_type,0,0
7,customer_type,0,0
8,reservation_status,0,0


In [ ]:
feature_reasoning = pd.DataFrame([
    [
        "total_nights",
        "stays_in_weekend_nights + stays_in_week_nights",
        "Represents the total length of the hotel stay."
    ],
    [
        "total_guests",
        "adults + children + babies",
        "Represents the total number of guests."
    ],
    [
        "total_special_requests_per_guest",
        "total_of_special_requests / total_guests",
        "Measures special-request intensity relative to group size."
    ],
    [
        "estimated_stay_revenue",
        "adr × total_nights",
        "Estimates room revenue from daily rate and stay length."
    ]
], columns=[
    "feature",
    "formula",
    "reasoning"
])

display(feature_reasoning)

,feature,formula,reasoning
0,total_nights,stays_in_weekend_nights + stays_in_week_nights,Represents the total length of the hotel stay.
1,total_guests,adults + children + babies,Represents the total number of guests.
2,total_special_requests_per_guest,total_of_special_requests / total_guests,Measures special-request intensity relative to...
3,estimated_stay_revenue,adr × total_nights,Estimates room revenue from daily rate and sta...


In [ ]:
def load_data():
    data = pd.read_csv(DATA_PATH)
    return data

In [ ]:
def clean(df):
    data = df.copy()

    data.columns = data.columns.str.strip()

    data["children"] = data["children"].fillna(
        data["children"].median()
    )

    data["country"] = data["country"].fillna(
        "Unknown"
    )

    data["agent"] = data["agent"].fillna(
        0
    ).astype(str)

    data["company"] = data["company"].fillna(
        0
    ).astype(str)

    categorical_columns = [
        "hotel",
        "meal",
        "market_segment",
        "distribution_channel",
        "reserved_room_type",
        "assigned_room_type",
        "deposit_type",
        "customer_type"
    ]

    for column in categorical_columns:
        data[column] = (
            data[column]
            .astype(str)
            .str.strip()
        )

    data["reservation_status_date"] = pd.to_datetime(
        data["reservation_status_date"],
        errors="coerce"
    )

    data["arrival_date"] = pd.to_datetime(
        data["arrival_date_year"].astype(str)
        + "-"
        + data["arrival_date_month"].astype(str)
        + "-"
        + data["arrival_date_day_of_month"].astype(str),
        errors="coerce"
    )

    data = data.drop_duplicates().reset_index(
        drop=True
    )

    data = data.drop(
        columns=[
            "reservation_status",
            "reservation_status_date"
        ],
        errors="ignore"
    )

    return data

In [ ]:
def add_features(df):
    data = df.copy()

    data["total_nights"] = (
        data["stays_in_weekend_nights"]
        + data["stays_in_week_nights"]
    )

    data["total_guests"] = (
        data["adults"].fillna(0)
        + data["children"].fillna(0)
        + data["babies"].fillna(0)
    )

    data["total_special_requests_per_guest"] = (
        data["total_of_special_requests"]
        / data["total_guests"].replace(0, np.nan)
    ).fillna(0)

    data["estimated_stay_revenue"] = (
        data["adr"]
        * data["total_nights"]
    )

    return data

In [ ]:
def build_pipeline(X):

    numeric_columns = X.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_columns = X.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    pipeline = ColumnTransformer([
        (
            "numeric",
            numeric_pipeline,
            numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ])

    return pipeline

In [ ]:
df_raw = load_data()

df_clean = clean(df_raw)

df_features = add_features(df_clean)

target = "is_canceled"

X = df_features.drop(
    columns=[target]
)

y = df_features[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

pipeline = build_pipeline(X_train)

X_train_processed = pipeline.fit_transform(
    X_train
)

X_test_processed = pipeline.transform(
    X_test
)

print("Raw data shape:", df_raw.shape)
print("Cleaned data shape:", df_clean.shape)
print("Feature-engineered data shape:", df_features.shape)

print("\nX_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nProcessed X_train:", X_train_processed.shape)
print("Processed X_test:", X_test_processed.shape)

Raw data shape: (119390, 32)
Cleaned data shape: (87396, 31)
Feature-engineered data shape: (87396, 35)

X_train: (69916, 34)
X_test: (17480, 34)

y_train: (69916,)
y_test: (17480,)

Processed X_train: (69916, 902)
Processed X_test: (17480, 902)


In [ ]:
assert not np.isnan(
    X_train_processed
).any()

assert not np.isnan(
    X_test_processed
).any()

assert X_train_processed.shape[1] == X_test_processed.shape[1]

assert X_train_processed.shape[0] == X_train.shape[0]

assert X_test_processed.shape[0] == X_test.shape[0]

print("All assertions passed successfully.")
print("No missing values remain in the processed matrices.")
print("Training and testing feature counts match.")

All assertions passed successfully.
No missing values remain in the processed matrices.
Training and testing feature counts match.


In [ ]:
before_after = pd.DataFrame([
    [
        "Raw",
        len(df_raw),
        df_raw.shape[1],
        int(df_raw.isna().sum().sum()),
        int(df_raw.duplicated().sum())
    ],
    [
        "Cleaned + Features",
        len(df_features),
        df_features.shape[1],
        int(df_features.isna().sum().sum()),
        int(df_features.duplicated().sum())
    ]
], columns=[
    "stage",
    "row_count",
    "column_count",
    "missing_values",
    "duplicate_rows"
])

display(before_after)

,stage,row_count,column_count,missing_values,duplicate_rows
0,Raw,119390,32,129425,31994
1,Cleaned + Features,87396,35,0,258


In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/ML Assignment 4"

# Create the directory if it does not exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLEANED_CSV_PATH = os.path.join(
    OUTPUT_DIR,
    "hotel_bookings_cleaned.csv"
)

PIPELINE_PATH = os.path.join(
    OUTPUT_DIR,
    "pipeline.joblib"
)

df_features.to_csv(
    CLEANED_CSV_PATH,
    index=False
)

joblib.dump(
    pipeline,
    PIPELINE_PATH
)

print("Cleaned CSV saved:")
print(CLEANED_CSV_PATH)

print("\nPipeline saved:")
print(PIPELINE_PATH)

Cleaned CSV saved:
/content/drive/MyDrive/ML Assignment 4/hotel_bookings_cleaned.csv

Pipeline saved:
/content/drive/MyDrive/ML Assignment 4/pipeline.joblib


In [ ]:
decision_report = pd.DataFrame([
    [
        "children",
        "Median imputation",
        int(df_raw["children"].isna().sum()),
        "Only a small number of values were missing."
    ],
    [
        "country",
        "Replace with Unknown",
        int(df_raw["country"].isna().sum()),
        "Missing country information is retained without inventing a country."
    ],
    [
        "agent",
        "Replace missing values with 0 and treat as categorical",
        int(df_raw["agent"].isna().sum()),
        "Agent is an identifier rather than a continuous measurement."
    ],
    [
        "company",
        "Replace missing values with 0 and treat as categorical",
        int(df_raw["company"].isna().sum()),
        "Company is an identifier rather than a continuous measurement."
    ],
    [
        "duplicate rows",
        "Remove duplicates",
        int(df_raw.duplicated().sum()),
        "Exact duplicate records do not provide additional information."
    ],
    [
        "reservation_status",
        "Remove",
        int(df_raw["reservation_status"].notna().sum()),
        "Final reservation outcome is removed to avoid outcome leakage."
    ],
    [
        "reservation_status_date",
        "Remove",
        int(df_raw["reservation_status_date"].notna().sum()),
        "Final reservation-status information is removed with the outcome field."
    ]
], columns=[
    "field",
    "decision",
    "count",
    "reason"
])

display(decision_report)

,field,decision,count,reason
0,children,Median imputation,4,Only a small number of values were missing.
1,country,Replace with Unknown,488,Missing country information is retained withou...
2,agent,Replace missing values with 0 and treat as cat...,16340,Agent is an identifier rather than a continuou...
3,company,Replace missing values with 0 and treat as cat...,112593,Company is an identifier rather than a continu...
4,duplicate rows,Remove duplicates,31994,Exact duplicate records do not provide additio...
5,reservation_status,Remove,119390,Final reservation outcome is removed to avoid ...
6,reservation_status_date,Remove,119390,Final reservation-status information is remove...


In [ ]:
print("""
Remaining limitations:

1. Some identifier fields such as agent and company do not represent
   continuous numerical measurements.

2. The dataset contains observational booking information and therefore
   may contain patterns or biases from the original booking process.

3. Missing values cannot always be recovered accurately because the
   original values are unavailable.

4. The engineered revenue feature is an estimate based on ADR and
   length of stay rather than an independently recorded revenue value.

5. The preprocessing creates a model-ready matrix, but preprocessing
   cannot remove all limitations or possible bias present in the raw data.
""")


Remaining limitations:

1. Some identifier fields such as agent and company do not represent
   continuous numerical measurements.

2. The dataset contains observational booking information and therefore
   may contain patterns or biases from the original booking process.

3. Missing values cannot always be recovered accurately because the
   original values are unavailable.

4. The engineered revenue feature is an estimate based on ADR and
   length of stay rather than an independently recorded revenue value.

5. The preprocessing creates a model-ready matrix, but preprocessing
   cannot remove all limitations or possible bias present in the raw data.



In [ ]:
print("=" * 60)
print("ASSIGNMENT 4 FINAL VERIFICATION")
print("=" * 60)

print("\nRaw dataset:")
print(df_raw.shape)

print("\nCleaned dataset:")
print(df_clean.shape)

print("\nFeature-engineered dataset:")
print(df_features.shape)

print("\nTraining data:")
print(X_train.shape)

print("\nTesting data:")
print(X_test.shape)

print("\nProcessed training matrix:")
print(X_train_processed.shape)

print("\nProcessed testing matrix:")
print(X_test_processed.shape)

print("\nMissing values in final dataset:")
print(int(df_features.isna().sum().sum()))

print("\nDuplicate rows in final dataset:")
print(int(df_features.duplicated().sum()))

print("\nCleaned CSV exists:")
print(os.path.exists(CLEANED_CSV_PATH))

print("\nPipeline exists:")
print(os.path.exists(PIPELINE_PATH))

assert os.path.exists(CLEANED_CSV_PATH)
assert os.path.exists(PIPELINE_PATH)

print("\nAssignment 4 preprocessing completed successfully.")

ASSIGNMENT 4 FINAL VERIFICATION

Raw dataset:
(119390, 32)

Cleaned dataset:
(87396, 31)

Feature-engineered dataset:
(87396, 35)

Training data:
(69916, 34)

Testing data:
(17480, 34)

Processed training matrix:
(69916, 902)

Processed testing matrix:
(17480, 902)

Missing values in final dataset:
0

Duplicate rows in final dataset:
258

Cleaned CSV exists:
True

Pipeline exists:
True

Assignment 4 preprocessing completed successfully.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

['health.pdf', 'migration - Page 1.pdf', 'driving licence of murari .pdf', 'amozon prime ppt', 'Colab Notebooks', 'Document from Murari Gangavarapu (5)', 'Document from Murari Gangavarapu (4)', 'Document from Murari Gangavarapu (3)', 'Document from Murari Gangavarapu (2)', 'Document from Murari Gangavarapu (1)', 'Unit 5_Java (1)_copy.pdf', 'Document from Murari Gangavarapu', 'Google AI Studio', 'map servey no. 146.pdf', 'Bhaji voice record.m4a', 'pan card of murari.pdf', 'aadhar card of murari (1).pdf', 'photo of murari.pdf', 'vasanth', 'letter of ministry - Line chart 1.gsheet', 'letter of ministry.gdoc', 'letter of municipal commissinor (1).gdoc', 'letter of municipal commissinor.gdoc', 'Classroom', 'Screenshot_20260407_215409.jpg', 'Adobe Scan Apr 07, 2026 (1).pdf', 'assignment (coa)_copy.pdf', 'Murari_CV_.pdf', 'aadhar card of murari.pdf', 'sem 3.pdf', 'sem 2.pdf', 'sem 1.pdf', 'sem 1_sem 2_sem 3_merged.pdf', 'NOC FORM.pdf', 'RSVP.gform', 'new health policy 1.pdf', 'new health poli

In [ ]:
DATA_PATH = "/content/drive/MyDrive/hotel_bookings.csv"

import pandas as pd

df_raw = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df_raw.shape)

Dataset loaded successfully.
Shape: (119390, 32)


In [ ]:
import os

assignment4_path = "/content/drive/MyDrive/ML Assignment 4"

print(os.listdir(assignment4_path))

['hotel_bookings_cleaned.csv', 'pipeline.joblib']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
